In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.rag.loader import load_documents

docs = load_documents("../data/docs/Maintenance_Manual.pdf")

In [5]:
len(docs)

92

In [6]:
docs[0]

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.4 (Windows)', 'creationdate': '2024-05-30T14:56:52+02:00', 'moddate': '2024-05-30T14:56:54+02:00', 'trapped': '/False', 'source': '../data/docs/Maintenance_Manual.pdf', 'total_pages': 92, 'page': 0, 'page_label': '1'}, page_content='Maintenance Manual')

In [7]:
from src.rag.chunker import split_documents

chunks = split_documents(docs)

In [8]:
len(chunks)

223

In [9]:
print(chunks[0].page_content)

Maintenance Manual


In [10]:
from src.rag.loader import load_documents
from src.rag.chunker import split_documents

docs = load_documents("../data/docs/Maintenance_Manual.pdf")

chunks = split_documents(docs)

print(f"Pages: {len(docs)}")
print(f"Chunks: {len(chunks)}")

Pages: 92
Chunks: 223


In [11]:
print(chunks[0].page_content)
print(chunks[0].metadata)

Maintenance Manual
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.4 (Windows)', 'creationdate': '2024-05-30T14:56:52+02:00', 'moddate': '2024-05-30T14:56:54+02:00', 'trapped': '/False', 'source': '../data/docs/Maintenance_Manual.pdf', 'total_pages': 92, 'page': 0, 'page_label': '1'}


In [12]:
from src.rag.embedder import get_embeddings

embeddings = get_embeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1434.96it/s]


In [13]:
vector = embeddings.embed_query("Bearing failure due to high temperature")

print(len(vector))

384


In [14]:
from src.rag.vector_store import create_vector_store
from src.rag.embedder import get_embeddings

embeddings = get_embeddings()

vector_store = create_vector_store(chunks, embeddings)

print(vector_store.index.ntotal)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2022.13it/s]


223


In [15]:
import importlib
import src.rag.vector_store as vs

importlib.reload(vs)

from src.rag.vector_store import create_vector_store

In [16]:
import src.rag.vector_store as vs

print(vs.__file__)

e:\AI Assistant for Predictive Maintenance\Ind_AI_Assistant\src\rag\vector_store.py


In [17]:
import inspect

print(inspect.getsource(vs.create_vector_store))

def create_vector_store(chunks, embeddings):
   

    vector_store = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    return vector_store



In [18]:
import importlib
import src.rag.vector_store as vs

importlib.reload(vs)

from src.rag.vector_store import create_vector_store

vector_store = create_vector_store(chunks, embeddings)

print(vector_store.index.ntotal)

223


In [19]:
import src.rag.vector_store as vs

print(vs.__file__)

e:\AI Assistant for Predictive Maintenance\Ind_AI_Assistant\src\rag\vector_store.py


....


In [20]:
from src.rag.retriever import get_retriever

retriever = get_retriever(vector_store)

In [21]:
docs = retriever.invoke(
    "How do I inspect bearings?"
)

In [22]:
len(docs)

3

In [23]:
print(docs[0].page_content)

determine when to change bearings in a unit. 
In the maintenance system, we handle this by connecting wireless sensors 
directly via "plug and play" to measure, for example, temperature, pressure or 
humidity. Through a simple configuration, you set a trigger for when you want to 
create a work order with an appropriate instruction for what to check or perform.
Condition checks can also be done by visual inspections or collecting data,


In [24]:
#botcheck llm only

from src.rag.chatbot import get_llm

llm = get_llm()

In [25]:
response = llm.invoke(
    "Explain predictive maintenance in one sentence."
)

print(response.content)

Predictive maintenance uses data to forecast equipment failures, allowing maintenance to be performed proactively before a breakdown occurs.


In [26]:
llm.invoke("What is predictive maintenance?")

AIMessage(content="**Predictive maintenance (PdM)** is a proactive maintenance strategy that uses data analysis and monitoring tools to predict when equipment is likely to fail, allowing maintenance to be performed *just before* a failure occurs.\n\nInstead of waiting for equipment to break down (reactive maintenance) or performing maintenance on a fixed schedule regardless of actual need (preventive maintenance), PdM aims to perform maintenance at the optimal time – when it's genuinely needed, but before it causes significant disruption or damage.\n\n### How Predictive Maintenance Works:\n\n1.  **Data Collection:** Sensors and monitoring devices are installed on critical equipment to continuously collect various types of data. This can include:\n    *   **Vibration analysis:** Detecting imbalances, misalignment, bearing wear, or loose components.\n    *   **Thermography (infrared imaging):** Identifying hot spots indicating electrical faults, friction, or insulation breakdown.\n    * 

In [27]:
llm = get_llm()

response = llm.invoke("What is predictive maintenance?")

print(response.content)

**Predictive maintenance (PdM)** is a proactive maintenance strategy that uses data-driven insights to predict when equipment is likely to fail, allowing maintenance to be performed *just before* a failure occurs.

Instead of waiting for equipment to break down (reactive maintenance) or performing maintenance on a fixed schedule regardless of actual need (preventive maintenance), PdM aims to optimize maintenance activities by addressing issues only when they are genuinely needed.

Here's a breakdown of what predictive maintenance entails:

1.  **Data Collection:**
    *   Sensors (IoT devices) are installed on critical equipment to continuously monitor various parameters.
    *   Common parameters include:
        *   **Vibration:** To detect imbalances, misalignments, bearing wear, gear damage.
        *   **Temperature:** To identify overheating, friction, electrical issues.
        *   **Acoustic (Ultrasound):** To detect leaks, arcing, early bearing degradation.
        *   **Oil A

In [28]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [29]:
docs = retriever.invoke(
    "How do I inspect bearings?"
)

context = format_docs(docs)

print(context)

determine when to change bearings in a unit. 
In the maintenance system, we handle this by connecting wireless sensors 
directly via "plug and play" to measure, for example, temperature, pressure or 
humidity. Through a simple configuration, you set a trigger for when you want to 
create a work order with an appropriate instruction for what to check or perform.
Condition checks can also be done by visual inspections or collecting data,

Maintenance Manual

• Vibration and oil analyses
• Thermography
• Spare parts management
• Documentation
• LCC (Life Cycle Cost)


In [30]:
question = "How do I inspect bearings?"

prompt = "You are an Industrial AI Assistant.Answer ONLY from the context."



print(prompt)

response = llm.invoke(prompt)

print(response.content)

You are an Industrial AI Assistant.Answer ONLY from the context.
Understood. I will answer ONLY from the context provided, acting as an Industrial AI Assistant.


In [31]:
#v2
from src.rag.chatbot import format_docs

docs = retriever.invoke("How do I inspect bearings?")

context = format_docs(docs)

print(context)

determine when to change bearings in a unit. 
In the maintenance system, we handle this by connecting wireless sensors 
directly via "plug and play" to measure, for example, temperature, pressure or 
humidity. Through a simple configuration, you set a trigger for when you want to 
create a work order with an appropriate instruction for what to check or perform.
Condition checks can also be done by visual inspections or collecting data,

Maintenance Manual

• Vibration and oil analyses
• Thermography
• Spare parts management
• Documentation
• LCC (Life Cycle Cost)


In [34]:
from src.rag.chatbot import ask_question

answer = ask_question(
    "How do I inspect bearings?",
    retriever
)

print(answer)

ImportError: cannot import name 'ask_question' from 'src.rag.chatbot' (e:\AI Assistant for Predictive Maintenance\Ind_AI_Assistant\src\rag\chatbot.py)

In [35]:
import importlib
import src.rag.chatbot as chatbot

importlib.reload(chatbot)

<module 'src.rag.chatbot' from 'e:\\AI Assistant for Predictive Maintenance\\Ind_AI_Assistant\\src\\rag\\chatbot.py'>

In [36]:
from src.rag.chatbot import ask_question

In [37]:
import src.rag.chatbot as chatbot

print(dir(chatbot))

['ChatGoogleGenerativeAI', 'ChatPromptTemplate', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'ask_question', 'format_docs', 'get_llm', 'get_prompt', 'load_dotenv', 'os']


In [38]:
from src.rag.chatbot import ask_question

answer = ask_question(
    "How do I inspect bearings?",
    retriever
)

print(answer)

You can inspect bearings through:
*   Visual inspections
*   Collecting data (such as temperature, pressure, or humidity measurements from wireless sensors)
*   Vibration and oil analyses
*   Thermography
